# DDPM Example Usage

This script walks through the full workflow:
setup → train → save/load → generate → plot.
Make sure the `ddpm/` package folder is in the same directory (or on your PYTHONPATH).

cd "C:\Users\lodik\Documents\programming\diffusion-models-project-main"

git add -A
git commit -m "Overwrite repo with local files"
git push

In [1]:
# --- Setup repo in Colab ---
%cd /content

import os
import sys

repo_path = "/content/ddpm-project-lodi"

if not os.path.exists(repo_path):
    !git clone https://github.com/LodiHendrikKamman/ddpm-project-lodi.git

%cd /content/ddpm-project-lodi
!git pull
!pip install -r requirements.txt

# Make sure Python can find the repo/package
if repo_path not in sys.path:
    sys.path.append(repo_path)

/content
/content/ddpm-project-lodi
remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 273.43 KiB | 6.51 MiB/s, done.
From https://github.com/LodiHendrikKamman/ddpm-project-lodi
   379a03c..69d5c45  main       -> origin/main
Updating 379a03c..69d5c45
Fast-forward
 lk_compare_time_att.ipynb   | 6427 +++++++++++++++++++++++++++++++++++++
 lk_example_usage_ddpm.ipynb | 7325 +++++++++++++++++++++++++++++++++++++++++--
 lk_example_usage_ddpm.py    |  136 -
 3 files changed, 13425 insertions(+), 463 deletions(-)
 create mode 100644 lk_compare_time_att.ipynb
 delete mode 100644 lk_example_usage_ddpm.py


In [2]:
# --- Imports ---
import torch

from ddpm import NoiseScheduler, UNet, train, find_lr, generate_image, noisy_image
from ddpm.dataset import load_mnist, get_noisy_loaders
from ddpm.utils import load_unet, channel_list, model_name, path_name
from ddpm.viz import plot_generated
import random
import numpy as np

# --- Expected behaviouring ---
seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# --- Device ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print("Imports worked!")

Using device: cuda
Imports worked!


## 1. Noise scheduler and data

The scheduler defines the beta schedule and handles the forward noising process.
T=1000 steps, linearly spaced betas from 1e-4 to 0.02 (Ho et al. defaults).

In [3]:
scheduler = NoiseScheduler(T=1000, beta_start=1e-4, beta_end=0.02).to(device)
train_set, test_set = load_mnist()

In [4]:
def zero_init_time_projections(model):
    """
    Makes the time-conditioning path start as a no-op.
    This was used in the good ablation runs.
    """
    count = 0

    for attr_name in ["encoder_time_projs", "decoder_time_projs"]:
        if hasattr(model, attr_name):
            proj_blocks = getattr(model, attr_name)

            for block in proj_blocks:
                for proj in block:
                    if isinstance(proj, torch.nn.Linear):
                        torch.nn.init.zeros_(proj.weight)
                        torch.nn.init.zeros_(proj.bias)
                        count += 1

    print(f"Zero-initialized {count} time projection layers.")

### Training on a single digit

Pass a filter function to get_noisy_loaders_filtered to restrict the dataset.
Here we train on zeros only, but any (dataset -> Subset) function works.

In [6]:
from ddpm.dataset import get_noisy_loaders_filtered, zeros_only

# unet_zeros = load_unet( "zeros_only.pkl", channels=channel_list(64), convs_per_level=2, num_heads_att=4, time_emb_dim=128, time_emb_base_dim=32,)

train_loader_zeros, test_loader_zeros = get_noisy_loaders_filtered( train_set, test_set, scheduler, filter_fn=zeros_only, batch_size=32)
unet_zeros = UNet( channels=channel_list(64), convs_per_level=2, num_heads_att=4, time_emb_dim=128, time_emb_base_dim=32,).to(device)
zero_init_time_projections(unet_zeros)
train_losses, test_losses = train( unet_zeros, train_loader_zeros, test_loader_zeros, epochs=50, lr=2e-4, weight_decay=1e-6,early_stopping_patience=10, save_path='zeros_only.pkl', use_time=True)

seed_sample = 999
random.seed(seed_sample)
np.random.seed(seed_sample)
torch.manual_seed(seed_sample)
torch.cuda.manual_seed_all(seed_sample)

x = generate_image(unet_zeros, scheduler, n_images=8, stochasticity=1.0, use_time=True,)
plot_generated(x, ncol=4)

Zero-initialized 10 time projection layers.


Epochs:   0%|          | 0/50 [00:00<?, ?it/s]

train:   0%|          | 0/186 [00:00<?, ?it/s]

test:   0%|          | 0/31 [00:00<?, ?it/s]

Epoch 0 | train loss: 0.2406 | test loss: 0.1118


train:   0%|          | 0/186 [00:00<?, ?it/s]

KeyboardInterrupt: 